# Model Evaluation & Explainability Notebook

This notebook contains the project pipeline: data preparation, baseline models, black-box models with explainability (SHAP/LIME), counterfactuals (DiCE), fairness auditing, comparative evaluation, and LaTeX export utilities. The code cells are split by section for easier execution and iterative development.

## Contents

- 1. Setup & Imports
- 2. Data Preparation & Cleaning
- 3. Baseline Models
- 4. Black-box Models + Explainability
- 5. Fairness Audit & Mitigation
- 6. Comparative Evaluation
- 7. Framework & Reporting
- How to run

## 1. Setup & Imports

In [ ]:
# Setup & Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE

# Metrics
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

# Explainability
import shap
from lime.lime_tabular import LimeTabularExplainer

# Fairness
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

# Counterfactuals
import dice_ml
from dice_ml import Dice

# Statistical tests
from scipy.stats import wilcoxon

# LaTeX export utilities
os.makedirs("latex_exports", exist_ok=True)

def save_plot_as_pdf(fig, filename):
    fig.savefig(f"latex_exports/{filename}.pdf", bbox_inches="tight")

def save_table_as_tex(df, filename):
    with open(f"latex_exports/{filename}.tex", "w") as f:
        f.write(df.to_latex(index=False, float_format="%.3f"))

## 2. Data Preparation & Cleaning (Weeks 1–2)

In [ ]:
# Data loading and basic cleaning
df = pd.read_csv("data/german_credit.csv")  # replace with your dataset path
print("Raw dataset shape:", df.shape)

# Handle missing values
for col in df.select_dtypes(include=['float64','int64']).columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Encode categorical
df_encoded = pd.get_dummies(df, drop_first=True)

# Scale features
X = df_encoded.drop("target", axis=1).values
y = df_encoded["target"].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Handle imbalance
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_scaled, y)
print("Balanced dataset shape:", X_res.shape)

## 3. Baseline Models (Weeks 3–4)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline_auc = []

for train_idx, test_idx in skf.split(X_res, y_res):
    X_train, X_test = X_res[train_idx], X_res[test_idx]
    y_train, y_test = y_res[train_idx], y_res[test_idx]
    
    lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    dt = DecisionTreeClassifier(max_depth=5).fit(X_train, y_train)
    
    for model in [lr, dt]:
        preds = model.predict_proba(X_test)[:,1]
        auc = roc_auc_score(y_test, preds)
        baseline_auc.append(auc)
        print(model.__class__.__name__, "AUC:", auc)

print("Baseline AUC variance:", np.var(baseline_auc))

## 4. Black-box Models + Explainability (Weeks 5–6)

In [ ]:
# Train a random forest as an example black-box model
rf = RandomForestClassifier(n_estimators=100).fit(X_res, y_res)

# SHAP global explanation
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_res)
fig, ax = plt.subplots()
shap.summary_plot(shap_values[1], X_res, feature_names=df_encoded.drop("target", axis=1).columns, show=False)
save_plot_as_pdf(fig, "shap_summary")

# LIME local explanation
lime_explainer = LimeTabularExplainer(X_res, mode="classification",
                                      training_labels=y_res,
                                      feature_names=df_encoded.drop("target", axis=1).columns)
i = 0
exp = lime_explainer.explain_instance(X_res[i], rf.predict_proba)
# Note: show_in_notebook may require a running Jupyter frontend
# exp.show_in_notebook(show_table=True)

# Counterfactuals with DiCE
df_cf = pd.DataFrame(X_res, columns=df_encoded.drop("target", axis=1).columns)
df_cf["target"] = y_res
d = dice_ml.Data(dataframe=df_cf,
                 continuous_features=df_encoded.drop("target", axis=1).columns.tolist(),
                 outcome_name="target")
m = dice_ml.Model(model=rf, backend="sklearn")
dice = Dice(d, m, method="random")
query_instance = df_cf.iloc[0].drop("target").to_dict()
dice_exp = dice.generate_counterfactuals(query_instance, total_CFs=3, desired_class="opposite")
cf_df = dice_exp.visualize_as_dataframe()
save_table_as_tex(cf_df, "counterfactuals")

## 5. Fairness Audit & Mitigation (Weeks 7–8)

In [ ]:
# Fairness metrics (example using a 'gender' column if present)
protected_attr = df["gender"].values if "gender" in df.columns else np.random.choice([0,1], size=len(y_res))
preds = rf.predict(X_res)

dp = demographic_parity_difference(y_res, preds, sensitive_features=protected_attr)
eo = equalized_odds_difference(y_res, preds, sensitive_features=protected_attr)

fairness_df = pd.DataFrame([{"Metric":"Demographic Parity Difference","Value":dp},{"Metric":"Equalized Odds Difference","Value":eo}])
save_table_as_tex(fairness_df, "fairness_metrics")

print("Demographic Parity Difference:", dp)
print("Equalized Odds Difference:", eo)

## 6. Comparative Evaluation (Weeks 9–10)

In [ ]:
results = []
for model in [lr, dt, rf]:
    preds = model.predict(X_res)
    acc = accuracy_score(y_res, preds)
    f1 = f1_score(y_res, preds)
    results.append({"model": model.__class__.__name__, "acc": acc, "f1": f1})

results_df = pd.DataFrame(results)
print(results_df)
save_table_as_tex(results_df, "comparative_results")

# Statistical test
scores = results_df["acc"].values
stat, p = wilcoxon(scores - np.mean(scores))
print("Wilcoxon test statistic:", stat, "p-value:", p)

sns.barplot(data=results_df, x="model", y="acc")
plt.title("Comparative Accuracy")
save_plot_as_pdf(plt.gcf(), "comparative_accuracy")
plt.show()

## 7. Framework & Reporting (Weeks 11–12)

In [ ]:
results_df.to_csv("results/final_report.csv", index=False)
print("Final framework and LaTeX-ready exports saved in 'latex_exports/' folder.")

## How to run

1. Install dependencies (create a virtual environment first). A minimal requirements list is included in `requirements.txt`.
2. Open this file in VS Code or Jupyter and run cells sequentially.
3. Note: some explainability/counterfactual visualizations require a running Jupyter frontend (e.g., `exp.show_in_notebook`).

If you want, I can also create a `requirements.txt` and a short `README.md` — say so and I will add them.